# 05-1. 最適化の基本 — 動かして確かめる

📖 解説: [`../01_basic_concepts.md`](../01_basic_concepts.md)

## このノートで触るもの
1. 勾配ゼロが極値の必要条件
2. 凸関数 vs 非凸関数の可視化
3. ヘッセ行列で点の種類を判定 (最小・最大・鞍点)
4. 局所最適 vs 大域最適

> 🧭 **クイックナビ**: 📚 [ROOT (全体 TOP)](../../README.md) ・ 🏠 [章 TOP](../README.md) ・ 📖 [解説 md (01_basic_concepts.md)](../01_basic_concepts.md)

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import japanize_matplotlib  # noqa: F401  # 日本語フォント (豆腐化対策)
from mpl_toolkits.mplot3d import Axes3D

%matplotlib inline

## 1. 勾配ゼロが極値の必要条件

$$
\nabla f(\mathbf{x}^*) = \mathbf{0} \quad \text{(極値の必要条件)}
$$

$f(\mathbf{x}) = (x_0 - 1)^2 + (x_1 - 2)^2$ の最小点は $(1, 2)$

In [ ]:
def f(x):
    return (x[0] - 1)**2 + (x[1] - 2)**2

grad_f = jax.grad(f)

for point in [[1.0, 2.0], [0.0, 0.0], [3.0, 4.0]]:
    p = jnp.array(point)
    print(f'∇f({point}) = {grad_f(p)}')

## 2. 凸関数 vs 非凸関数の可視化

凸関数: お椀型、局所=大域最適

非凸関数: 複数の局所最適、注意

In [ ]:
x = np.linspace(-3, 3, 200)

# 凸関数: f(x) = x²
y_convex = x**2

# 非凸関数: f(x) = x⁴ - 4x² + x + 5 (2つの谷)
y_nonconvex = x**4 - 4*x**2 + x + 5

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(x, y_convex, 'b-', linewidth=2)
axes[0].set_title('凸関数: $f(x) = x^2$\n→ 局所最適 = 大域最適')
axes[0].axvline(0, color='red', linestyle='--', alpha=0.5)
axes[0].grid(True, alpha=0.3)

axes[1].plot(x, y_nonconvex, 'g-', linewidth=2)
axes[1].set_title('非凸関数: $f(x) = x^4 - 4x^2 + x + 5$\n→ 局所最適が複数')
axes[1].grid(True, alpha=0.3)

# 局所最適と大域最適
from scipy.optimize import minimize
for x0 in [-2.5, 2.5]:
    res = minimize(lambda v: v**4 - 4*v**2 + v + 5, x0)
    axes[1].plot(res.x, res.fun, 'ro', markersize=12)
    axes[1].annotate(f'x={res.x[0]:.2f}\nf={res.fun:.2f}',
                     (res.x[0], res.fun), xytext=(10, 10), textcoords='offset points')

plt.tight_layout()
plt.show()

## 3. ヘッセ行列の固有値で点の種類を判定

$$
H_f = \begin{pmatrix} \frac{\partial^2 f}{\partial x_1^2} & \frac{\partial^2 f}{\partial x_1 \partial x_2} \\ \frac{\partial^2 f}{\partial x_2 \partial x_1} & \frac{\partial^2 f}{\partial x_2^2} \end{pmatrix}
$$

| 固有値 | 種類 |
|---|---|
| 全部正 | 最小点 (お椀) |
| 全部負 | 最大点 (山) |
| 正負混在 | 鞍点 |

In [ ]:
test_funcs = {
    '凸 (お椀): x² + y²':       lambda p: p[0]**2 + p[1]**2,
    '凸 (歪んだ): x² + 3y²':    lambda p: p[0]**2 + 3*p[1]**2,
    '凹 (山): -(x² + y²)':       lambda p: -(p[0]**2 + p[1]**2),
    '鞍点: x² - y²':             lambda p: p[0]**2 - p[1]**2,
}

for name, f in test_funcs.items():
    H = jax.hessian(f)(jnp.array([0.0, 0.0]))
    eigs = jnp.linalg.eigvalsh(H)
    if jnp.all(eigs > 0):
        verdict = '✅ 強凸 (最小点)'
    elif jnp.all(eigs < 0):
        verdict = '⚠️ 凹 (最大点)'
    else:
        verdict = '❌ 鞍点 (要注意)'
    print(f'{name}')
    print(f'  Hessian 固有値: {np.array(eigs)}')
    print(f'  判定: {verdict}\n')

## 4. 鞍点を 3D で可視化

$$
f(x, y) = x^2 - y^2
$$

$x$ 方向は谷、$y$ 方向は山 → 鞍点。ML 訓練で問題になるケース。

In [ ]:
x = np.linspace(-2, 2, 50)
y = np.linspace(-2, 2, 50)
X, Y = np.meshgrid(x, y)

fig = plt.figure(figsize=(15, 5))
for i, (Z, title) in enumerate([
    (X**2 + Y**2, '凸 (お椀): $x^2 + y^2$'),
    (-(X**2 + Y**2), '凹 (山): $-(x^2 + y^2)$'),
    (X**2 - Y**2, '鞍点: $x^2 - y^2$'),
], 1):
    ax = fig.add_subplot(1, 3, i, projection='3d')
    ax.plot_surface(X, Y, Z, cmap='coolwarm', alpha=0.7)
    ax.set_title(title)
    ax.set_xlabel('x'); ax.set_ylabel('y')
plt.tight_layout()
plt.show()

## 5. 強凸性のヘッセ行列条件

強凸 ⟺ ヘッセ行列が正定値 (全固有値 > 0)

$$
f(\mathbf{y}) \geq f(\mathbf{x}) + \nabla f(\mathbf{x})^\top (\mathbf{y} - \mathbf{x}) + \frac{\mu}{2} \|\mathbf{y} - \mathbf{x}\|^2
$$

In [ ]:
def f(x):
    return x[0]**2 + 2*x[1]**2 + 0.1 * x[0] * x[1]

H = jax.hessian(f)(jnp.array([1.0, 1.0]))
eigvals = jnp.linalg.eigvalsh(H)
mu = float(eigvals.min())   # 強凸性パラメータ
L = float(eigvals.max())    # 平滑性パラメータ

print(f'ヘッセ行列:\n{np.array(H)}')
print(f'\n固有値: {np.array(eigvals)}')
print(f'強凸性パラメータ μ = {mu:.4f}')
print(f'平滑性パラメータ L = {L:.4f}')
print(f'条件数 κ = L / μ = {L/mu:.4f}  ← 大きいと収束が遅い')

## まとめ
- 極値の必要条件: $\nabla f = \mathbf{0}$
- 凸関数なら局所=大域最適、勾配降下で確実に到達
- ヘッセ行列の固有値で点の種類が判定できる
- 鞍点は ML 訓練で問題 → モメンタム等で脱出

## 次へ
→ [`02_gradient_descent.ipynb`](02_gradient_descent.ipynb) — 勾配降下法の実装

解説 → [`../02_gradient_descent.md`](../02_gradient_descent.md)

---

## 📍 ナビゲーション

| ← 前 | 🏠 章 TOP | 📚 全体 TOP | 次 → |
|---|---|---|---|
| [章 TOP](../README.md) | [章 TOP](../README.md) | [📚 ROOT README](../../README.md) | [`02_gradient_descent.ipynb`](02_gradient_descent.ipynb) |